## 1. Load data

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('wines_clean.csv')

In [3]:
df.head(5)

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery,description_clean
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,25.0,Sicily & Sardinia,Etna,Unknown,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia,aromas include tropical fruit broom brimstone ...
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,Unknown,Unknown,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos,this is ripe and fruity a wine that is smooth ...
2,US,"Tart and snappy, the flavors of lime flesh and...",Unknown,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm,tart and snappy the flavors of lime flesh and ...
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,Unknown,Alexander Peartree,Unknown,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian,pineapple rind lemon pith and orange blossom s...
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks,much like the regular bottling from this comes...


In [4]:
SAMPLE_SIZE = 3000
df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

In [5]:
df.shape

(3000, 14)

In [6]:
df[["title", "variety"]].head(3)

,title,variety
0,Michel Reybier NV Brut Premier Cru (Champagne),Champagne Blend
1,Leone de Castris 2011 Five Roses Rosato Negroa...,Negroamaro
2,Vine Cliff 2004 Pickett Road Cabernet Sauvigno...,Cabernet Sauvignon


## 2. Embeddings with SBERT

In [ ]:
#!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 4.0 MB/s  0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer


2026-07-18 12:04:34.194476: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-18 12:04:34.207903: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-18 12:04:34.317663: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-18 12:04:34.463741: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-18 12:04:34.588497: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(
    df["description"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

In [14]:
embeddings.dtype

dtype('float32')

In [15]:
print("Shape of the matrix:", embeddings.shape)

Shape of the matrix: (3000, 384)


In [16]:
print("First 10 numbers of the embedding of wine 0:")
print(embeddings[0][:10])
print()
print("Wine 0 is:", df.loc[0, "title"])
print()
# confirm the vector is normalized (length very close to 1)
print("Length of the vector:", np.linalg.norm(embeddings[0]))

First 10 numbers of the embedding of wine 0:
[ 0.01438079 -0.01942611 -0.01228239  0.02226519  0.00475127  0.02502213
 -0.01199461 -0.00162239 -0.02003952 -0.09481858]

Wine 0 is: Michel Reybier NV Brut Premier Cru  (Champagne)

Length of the vector: 1.0


## 3. Cosine similarity

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
query_embedding = model.encode(
    ["fruity wine"],
    normalize_embeddings=True
)


In [37]:
scores = cosine_similarity(
    query_embedding,
    embeddings
)[0]

print(scores)

[0.49839342 0.43970767 0.36959654 ... 0.5514382  0.43216228 0.6088208 ]


In [38]:
order = np.argsort(scores)[::-1]

top5 = order[:5]

for i in top5:
    print(df.iloc[i]["title"], scores[i])

Adega Cooperativa de Borba 2015 Convento da Vila Red (Alentejano) 0.77162385
Maison Ginestet 2015 Clairet des Carrelets Red (Bordeaux Clairet) 0.77013946
Domaine Chasselay 2010 Quatre Saisons  (Beaujolais) 0.759773
Hogue 2008 Genesis Syrah (Columbia Valley (WA)) 0.7557086
Cortes de Cima 2008 Trincadeira (Alentejano) 0.75472856


In [39]:
top5_df = df.iloc[top5][["title", "description"]].copy()

top5_df["similarity"] = scores[top5]

print(top5_df)

                                                  title  \
2546  Adega Cooperativa de Borba 2015 Convento da Vi...   
655   Maison Ginestet 2015 Clairet des Carrelets Red...   
2098  Domaine Chasselay 2010 Quatre Saisons  (Beaujo...   
2156    Hogue 2008 Genesis Syrah (Columbia Valley (WA))   
256        Cortes de Cima 2008 Trincadeira (Alentejano)   

                                            description  similarity  
2546  This broad, fruity wine brings out ripe berry ...    0.771624  
655   Fruity and citrus flavored, this is a wine tha...    0.770139  
2098  A warm and rich wine that shows soft, ready-to...    0.759773  
2156  A fruity wine, with tangy cranberry and raspbe...    0.755709  
256   A ripe, lively wine that manages to restrain i...    0.754729  
